# CMIP6 XHWI Monthly Accumulated Index

This notebook computes the monthly accumulated Extreme Heatwave Index (XHWI) for CMIP6 data.

The workflow follows the ERA5 implementation logic:

- use daily `tasmax` from 1961-1990 as the calibration period;
- compute one empirical CDF per calendar month and grid cell;
- interpolate 3-hourly `tas` and `huss` to hourly resolution;
- convert `huss` to relative humidity assuming standard surface pressure;
- match hourly temperature values to the calibration CDF;
- compute hourly XHWI;
- aggregate to daily and then monthly accumulated values;
- write one CF-oriented NetCDF file per scenario.

Each code cell is organized as a future script/module boundary to simplify migration from notebook to a script-based CMIP6 module.

## Future File: `cmip6/spatial/notebooks/setup_colab.py`

In [ ]:
from google.colab import drive
drive.mount('/content/drive/')

import os

user = input('Who is? ')
target_dir = 'drive/My Drive/Mestrado/lammoc/indices'

if user.upper() in {'LIVIA', 'VITOR'}:
    if os.getcwd() == '/content':
        os.chdir(target_dir)
    print(os.getcwd())
    print(os.listdir())
else:
    raise ValueError("Expected user to be 'LIVIA' or 'VITOR'.")

## Future File: `cmip6/spatial/notebooks/install_dependencies.py`

In [ ]:
!pip -q install zarr dask distributed numcodecs cftime netCDF4 h5netcdf scipy
!pip -q install xclim climate-indices

## Future File: `cmip6/spatial/scripts/src/config/settings.py`

In [ ]:
from pathlib import Path

BASE_DIR = Path.cwd()
MODEL_ID = 'BCC-CSM2-MR'
GRID_LABEL = 'gn'
MEMBER_ID = 'r1i1p1f1'
CALIBRATION_PERIOD = ('1961-01-01', '1990-12-31')

if (BASE_DIR / MODEL_ID).exists():
    CMIP6_ROOT = BASE_DIR
elif (BASE_DIR / 'cmip6' / MODEL_ID).exists():
    CMIP6_ROOT = BASE_DIR / 'cmip6'
elif (BASE_DIR / 'index-xhwi' / 'cmip6' / MODEL_ID).exists():
    CMIP6_ROOT = BASE_DIR / 'index-xhwi' / 'cmip6'
else:
    raise FileNotFoundError(f'Could not locate CMIP6 root from {BASE_DIR}')

MODEL_ROOT = CMIP6_ROOT / MODEL_ID
OUTPUT_DIR = MODEL_ROOT / 'results' / 'xhwi'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f'CMIP6 root: {CMIP6_ROOT}')
print(f'Model root: {MODEL_ROOT}')

SCENARIOS = {
    'historical': {
        'tas': MODEL_ROOT / 'historical' / '3hr' / 'tas' / GRID_LABEL / f'member-{MEMBER_ID}.zarr',
        'huss': MODEL_ROOT / 'historical' / '3hr' / 'huss' / GRID_LABEL / f'member-{MEMBER_ID}.zarr',
        # Historical daily tasmax is currently stored as ensemble_mean.zarr in this dataset layout.
        'tasmax': MODEL_ROOT / 'historical' / 'day' / 'tasmax' / GRID_LABEL / 'ensemble_mean.zarr',
    },
    'ssp245': {
        'tas': MODEL_ROOT / 'ssp245' / '3hr' / 'tas' / GRID_LABEL / f'member-{MEMBER_ID}.zarr',
        'huss': MODEL_ROOT / 'ssp245' / '3hr' / 'huss' / GRID_LABEL / f'member-{MEMBER_ID}.zarr',
        'tasmax': MODEL_ROOT / 'ssp245' / 'day' / 'tasmax' / GRID_LABEL / f'member-{MEMBER_ID}.zarr',
    },
    'ssp585': {
        'tas': MODEL_ROOT / 'ssp585' / '3hr' / 'tas' / GRID_LABEL / f'member-{MEMBER_ID}.zarr',
        'huss': MODEL_ROOT / 'ssp585' / '3hr' / 'huss' / GRID_LABEL / f'member-{MEMBER_ID}.zarr',
        'tasmax': MODEL_ROOT / 'ssp585' / 'day' / 'tasmax' / GRID_LABEL / f'member-{MEMBER_ID}.zarr',
    },
}

TEMPERATURE_THRESHOLD_C = 32.0
CDF_THRESHOLD_PERCENT = 95.0
STANDARD_PRESSURE_PA = 101325.0
TIME_CHUNK_HOURLY = 24 * 31
SPATIAL_CHUNK = 32

for scenario, paths in SCENARIOS.items():
    for name, path in paths.items():
        print(f'{scenario:10s} {name:6s}: {path}')

## Future File: `cmip6/spatial/scripts/src/utils/imports.py`

In [ ]:
from datetime import datetime, timezone

import numpy as np
import xarray as xr
import dask
from dask.diagnostics import ProgressBar

dask.config.set({
    'array.slicing.split_large_chunks': False,
    'optimization.fuse.active': True,
})

## Future File: `cmip6/spatial/scripts/src/preprocessing/cmip6.py`

In [ ]:
def clean_cmip6_dims(ds: xr.Dataset) -> xr.Dataset:
    """Keep only time, lat, and lon dimensions and sort them."""
    keep_dims = {'time', 'lat', 'lon'}

    vars_to_keep = [
        var for var in ds.data_vars
        if set(ds[var].dims).issubset(keep_dims)
    ]
    ds = ds[vars_to_keep]

    coords_to_drop = [coord for coord in ds.coords if coord not in keep_dims]
    ds = ds.drop_vars(coords_to_drop, errors='ignore')

    for dim in ['time', 'lat', 'lon']:
        if dim in ds.dims:
            ds = ds.sortby(dim)

    return ds


def open_clean_zarr(path: Path, chunks: dict | None = None) -> xr.Dataset:
    ds = xr.open_zarr(path, chunks=chunks)
    return clean_cmip6_dims(ds)


def interpolate_to_hourly(ds: xr.Dataset) -> xr.Dataset:
    return ds.sortby('time').resample(time='1h').interpolate('linear')


def kelvin_to_celsius(da: xr.DataArray) -> xr.DataArray:
    units = str(da.attrs.get('units', '')).lower()
    out = da - 273.15 if units in {'k', 'kelvin'} else da
    out = out.copy()
    out.attrs.update(da.attrs)
    out.attrs['units'] = 'degC'
    return out

## Future File: `cmip6/spatial/scripts/src/features/humidity.py`

In [ ]:
def specific_to_relative_humidity_standard_pressure(
    huss: xr.DataArray,
    tas: xr.DataArray,
    p0: float = STANDARD_PRESSURE_PA,
    clip: bool = True,
) -> xr.DataArray:
    """Convert specific humidity to relative humidity using standard pressure."""
    epsilon = 0.622
    tas_c = tas - 273.15 if str(tas.attrs.get('units', '')).lower() in {'k', 'kelvin'} else tas

    e = (huss * p0) / (epsilon + (1 - epsilon) * huss)
    es = 611.2 * np.exp((17.67 * tas_c) / (tas_c + 243.5))
    hurs = 100.0 * e / es

    if clip:
        hurs = hurs.clip(min=0, max=100)

    hurs.name = 'hurs'
    hurs.attrs.update({
        'standard_name': 'relative_humidity',
        'long_name': 'Relative humidity',
        'units': '%',
        'description': (
            'Relative humidity computed from specific humidity and air temperature '
            'assuming standard surface pressure p = 101325 Pa. Saturation vapor '
            'pressure computed using Bolton 1980.'
        ),
        'assumed_pressure': '101325 Pa',
    })
    return hurs

## Future File: `cmip6/spatial/scripts/src/cdf/cdf.py`

In [ ]:
def _sort_and_cdf_np(a_1d: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    a = np.asarray(a_1d)
    mask = np.isfinite(a)

    if not mask.any():
        n = a.size
        return np.full(n, np.nan), np.full(n, np.nan)

    valid = a[mask]
    sorted_valid = np.sort(valid)
    n = sorted_valid.size
    cdf_valid = np.arange(1, n + 1) / n

    sorted_full = np.full_like(a, np.nan, dtype=sorted_valid.dtype)
    cdf_full = np.full_like(a, np.nan, dtype=float)
    sorted_full[mask] = sorted_valid
    cdf_full[mask] = cdf_valid
    return sorted_full, cdf_full


def compute_sorted_and_cdf(da: xr.DataArray, dim: str = 'calibration_time') -> tuple[xr.DataArray, xr.DataArray]:
    sorted_vals, cdf_vals = xr.apply_ufunc(
        _sort_and_cdf_np,
        da,
        input_core_dims=[[dim]],
        output_core_dims=[[dim], [dim]],
        vectorize=True,
        dask='parallelized',
        dask_gufunc_kwargs={'allow_rechunk': True},
        output_dtypes=[da.dtype, float],
    )
    sorted_vals.name = 'sorted_tasmax'
    cdf_vals.name = 'tasmax_cdf'
    return sorted_vals, cdf_vals


def _interp_cdf_np(v_series: np.ndarray, sorted_series: np.ndarray, cdf_series: np.ndarray) -> np.ndarray:
    v = np.asarray(v_series)
    x = np.asarray(sorted_series)
    y = np.asarray(cdf_series)

    if not np.isfinite(x).any() or not np.isfinite(v).any():
        return np.full_like(v, np.nan, dtype=float)

    mask_xy = np.isfinite(x) & np.isfinite(y)
    if mask_xy.sum() < 2:
        return np.full_like(v, np.nan, dtype=float)

    out = np.interp(v, x[mask_xy], y[mask_xy], left=0.0, right=1.0)
    out[~np.isfinite(v)] = np.nan
    return out


def match_cdf(
    validation_tas: xr.DataArray,
    sorted_tasmax: xr.DataArray,
    cdf_vals: xr.DataArray,
    validation_dim: str = 'time',
    calibration_dim: str = 'calibration_time',
) -> xr.DataArray:
    target = xr.apply_ufunc(
        _interp_cdf_np,
        validation_tas,
        sorted_tasmax,
        cdf_vals,
        input_core_dims=[[validation_dim], [calibration_dim], [calibration_dim]],
        output_core_dims=[[validation_dim]],
        vectorize=True,
        dask='parallelized',
        dask_gufunc_kwargs={'allow_rechunk': True},
        output_dtypes=[float],
    )
    target.name = 'Target'
    target.attrs.update({
        'long_name': 'Calibration cumulative probability matched to hourly air temperature',
        'units': '1',
    })
    return target

## Future File: `cmip6/spatial/scripts/src/features/xhwi.py`

In [ ]:
def heatwave_index(tas_c: xr.DataArray, hurs: xr.DataArray, target: xr.DataArray) -> xr.DataArray:
    target100 = target * 100.0
    tpe = (target100 - CDF_THRESHOLD_PERCENT).clip(min=0)
    coef = (np.exp(tpe) * hurs) / 1000.0
    xhwi = (coef - 0.001) / 14.84

    xhwi = xhwi.where(tpe > 0, 0)
    xhwi = xhwi.where(tas_c > TEMPERATURE_THRESHOLD_C, 0)
    xhwi = xhwi.where(xhwi > 0.001, 0)

    xhwi.name = 'xhwi'
    xhwi.attrs.update({
        'long_name': 'Extreme Heatwave Index',
        'units': '1',
        'description': (
            'Hourly Extreme Heatwave Index computed from calibration CDF probability, '
            'air temperature, and relative humidity.'
        ),
    })
    return xhwi

## Future File: `cmip6/spatial/scripts/src/features/aggregations.py`

In [ ]:
def monthly_accumulated_xhwi(xhwi: xr.DataArray) -> xr.DataArray:
    active_hours = (xhwi != 0).astype('int16').resample(time='1D').sum()
    daily_sum = xhwi.resample(time='1D').sum()
    daily_indicator = active_hours * daily_sum
    daily_indicator.name = 'xhwi_daily_accumulated'

    monthly = daily_indicator.resample(time='MS').sum()
    monthly.name = 'xhwi_monthly_accumulated'
    monthly.attrs.update({
        'long_name': 'Monthly accumulated Extreme Heatwave Index',
        'units': '1',
        'cell_methods': 'time: sum',
        'description': (
            'Monthly sum of daily XHWI products. Each daily product is the number '
            'of hours with nonzero XHWI multiplied by the daily sum of hourly XHWI.'
        ),
    })
    return monthly

## Future File: `cmip6/spatial/scripts/src/io/writers.py`

In [ ]:
def build_monthly_output_dataset(monthly: xr.DataArray, scenario: str) -> xr.Dataset:
    ds = monthly.to_dataset(name='xhwi_monthly_accumulated')

    if 'lat' in ds.coords:
        ds['lat'].attrs.update({
            'standard_name': 'latitude',
            'long_name': 'Latitude',
            'units': 'degrees_north',
            'axis': 'Y',
        })
    if 'lon' in ds.coords:
        ds['lon'].attrs.update({
            'standard_name': 'longitude',
            'long_name': 'Longitude',
            'units': 'degrees_east',
            'axis': 'X',
        })
    if 'time' in ds.coords:
        ds['time'].attrs.update({
            'standard_name': 'time',
            'long_name': 'Time',
            'axis': 'T',
        })

    ds.attrs.update({
        'Conventions': 'CF-1.10',
        'title': f'Monthly accumulated XHWI for CMIP6 {MODEL_ID} {scenario}',
        'source': (
            f'CMIP6 model {MODEL_ID}, experiment {scenario}, member {MEMBER_ID}; '
            'XHWI computed from 3-hourly tas and huss interpolated to hourly resolution; '
            'calibration CDF computed from daily tasmax for 1961-1990.'
        ),
        'creator': 'LAMMOC-UFF',
        'contact': 'mcataldi@id.uff.br',
        'institution': 'Laboratory for Monitoring and Modeling of Climate Systems, Federal Fluminense University',
        'creation_date': datetime.now(timezone.utc).strftime('%Y-%m-%dT%H:%M:%SZ'),
        'model_id': MODEL_ID,
        'experiment_id': scenario,
        'member_id': MEMBER_ID,
        'grid_label': GRID_LABEL,
        'calibration_period': f'{CALIBRATION_PERIOD[0]} to {CALIBRATION_PERIOD[1]}',
        'input_variables': 'tas, huss, tasmax',
        'humidity_method': 'Specific humidity converted to relative humidity assuming p = 101325 Pa.',
        'temperature_threshold': f'{TEMPERATURE_THRESHOLD_C} degC',
        'cdf_threshold': f'p{int(CDF_THRESHOLD_PERCENT)}',
    })
    return ds


def write_monthly_netcdf(ds: xr.Dataset, scenario: str) -> Path:
    output_path = OUTPUT_DIR / f'xhwi_cmip6_{MODEL_ID}_{scenario}_{MEMBER_ID}_monthly_accumulated.nc'
    ds = ds.sortby('time')

    if 'time' in ds.indexes and not ds.indexes['time'].is_monotonic_increasing:
        raise ValueError('Output time coordinate is not monotonic increasing.')
    if 'time' in ds.indexes and not ds.indexes['time'].is_unique:
        raise ValueError('Output time coordinate contains duplicated values.')

    encoding = {
        'xhwi_monthly_accumulated': {
            'zlib': True,
            'complevel': 4,
            '_FillValue': np.float32(np.nan),
            'dtype': 'float32',
        }
    }

    with ProgressBar():
        ds.to_netcdf(output_path, engine='netcdf4', encoding=encoding)

    return output_path

## Future File: `cmip6/spatial/scripts/src/pipeline/calibration.py`

In [ ]:
def open_calibration_tasmax() -> xr.DataArray:
    tasmax_ds = open_clean_zarr(
        SCENARIOS['historical']['tasmax'],
        chunks={'time': -1, 'lat': SPATIAL_CHUNK, 'lon': SPATIAL_CHUNK},
    )
    tasmax = kelvin_to_celsius(tasmax_ds['tasmax'])
    tasmax = tasmax.sel(time=slice(*CALIBRATION_PERIOD))

    if tasmax.sizes.get('time', 0) == 0:
        raise ValueError(f'No tasmax data found for calibration period {CALIBRATION_PERIOD}.')

    return tasmax.rename({'time': 'calibration_time'})


def build_monthly_calibration_cdfs() -> dict[int, tuple[xr.DataArray, xr.DataArray]]:
    tasmax_calibration = open_calibration_tasmax()
    monthly_cdfs = {}

    for month in range(1, 13):
        tasmax_month = tasmax_calibration.sel(
            calibration_time=tasmax_calibration['calibration_time.month'] == month
        )

        if tasmax_month.sizes.get('calibration_time', 0) == 0:
            raise ValueError(f'No calibration tasmax data found for month {month}.')

        monthly_cdfs[month] = compute_sorted_and_cdf(tasmax_month, dim='calibration_time')

    return monthly_cdfs


monthly_calibration_cdfs = build_monthly_calibration_cdfs()
display(monthly_calibration_cdfs[1][0])
display(monthly_calibration_cdfs[1][1])

## Future File: `cmip6/spatial/scripts/src/pipeline/monthly_pipeline.py`

In [ ]:
def open_hourly_scenario_inputs(scenario: str) -> tuple[xr.DataArray, xr.DataArray]:
    paths = SCENARIOS[scenario]
    chunks = {'time': TIME_CHUNK_HOURLY, 'lat': SPATIAL_CHUNK, 'lon': SPATIAL_CHUNK}

    tas_ds = interpolate_to_hourly(open_clean_zarr(paths['tas'], chunks=chunks))
    huss_ds = interpolate_to_hourly(open_clean_zarr(paths['huss'], chunks=chunks))

    tas = tas_ds['tas']
    huss = huss_ds['huss']

    tas_c = kelvin_to_celsius(tas)
    hurs = specific_to_relative_humidity_standard_pressure(huss=huss, tas=tas)

    return tas_c, hurs


def compute_scenario_monthly_xhwi(scenario: str) -> xr.Dataset:
    print(f'Opening and preprocessing {scenario} inputs...')
    tas_c, hurs = open_hourly_scenario_inputs(scenario)

    monthly_outputs = []

    for month in range(1, 13):
        print(f'Processing {scenario}, calendar month {month:02d}...')
        tas_c_month = tas_c.sel(time=tas_c['time.month'] == month)
        hurs_month = hurs.sel(time=hurs['time.month'] == month)

        if tas_c_month.sizes.get('time', 0) == 0:
            print(f'No hourly tas data found for {scenario}, month {month:02d}; skipping.')
            continue

        sorted_tasmax, tasmax_cdf = monthly_calibration_cdfs[month]
        target = match_cdf(
            validation_tas=tas_c_month,
            sorted_tasmax=sorted_tasmax,
            cdf_vals=tasmax_cdf,
            validation_dim='time',
            calibration_dim='calibration_time',
        )

        xhwi = heatwave_index(tas_c=tas_c_month, hurs=hurs_month, target=target)
        monthly_month = monthly_accumulated_xhwi(xhwi)
        monthly_month = monthly_month.sel(time=monthly_month['time.month'] == month)
        monthly_outputs.append(monthly_month)

    if not monthly_outputs:
        raise ValueError(f'No monthly XHWI outputs were generated for {scenario}.')

    monthly = xr.concat(monthly_outputs, dim='time').sortby('time')
    monthly = monthly.chunk({'time': 120, 'lat': SPATIAL_CHUNK, 'lon': SPATIAL_CHUNK})

    return build_monthly_output_dataset(monthly, scenario=scenario)

## Run: Historical

In [ ]:
ds_historical_monthly = compute_scenario_monthly_xhwi('historical')
display(ds_historical_monthly)
historical_output = write_monthly_netcdf(ds_historical_monthly, 'historical')
print(historical_output)

## Run: SSP2-4.5

In [ ]:
ds_ssp245_monthly = compute_scenario_monthly_xhwi('ssp245')
display(ds_ssp245_monthly)
ssp245_output = write_monthly_netcdf(ds_ssp245_monthly, 'ssp245')
print(ssp245_output)

## Run: SSP5-8.5

In [ ]:
ds_ssp585_monthly = compute_scenario_monthly_xhwi('ssp585')
display(ds_ssp585_monthly)
ssp585_output = write_monthly_netcdf(ds_ssp585_monthly, 'ssp585')
print(ssp585_output)

## Optional CF Metadata Inspection

In [ ]:
# Quick metadata inspection. For a full CF check, run cfchecks externally if available.
for path in [
    OUTPUT_DIR / f'xhwi_cmip6_{MODEL_ID}_historical_{MEMBER_ID}_monthly_accumulated.nc',
    OUTPUT_DIR / f'xhwi_cmip6_{MODEL_ID}_ssp245_{MEMBER_ID}_monthly_accumulated.nc',
    OUTPUT_DIR / f'xhwi_cmip6_{MODEL_ID}_ssp585_{MEMBER_ID}_monthly_accumulated.nc',
]:
    if path.exists():
        ds = xr.open_dataset(path)
        print(path)
        print(ds)
        print(ds.attrs)
        print(ds['xhwi_monthly_accumulated'].attrs)
        ds.close()